# FactIndic 02 – Code-mixed input variants

An LLM rewrites every test (and LoRA training) article sentence by sentence into code-mixed text (native-script grammar, English content words in Latin letters, e.g. *भारत में pollution का level लगातार increase हो रहा है*), then transliterates that into fully romanised Hinglish/Banglish. Each sentence is checked (numbers unchanged, length, script) and retried once; articles with under 80% valid sentences are excluded from the paired comparison. The log prints the batch size and an estimated time left every few minutes. Also writes `codemix_quality_check.xlsx` for a quick human check of meaning and naturalness.

| Setting (right-hand panel) | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** |
| Inputs to attach | output of notebook 01 |
| Time (full profile) | about 1–2 h; the log prints an estimate |

Run it with **Save Version → Save & Run All (Commit)**. When it finishes, its output (folder `factindic_out`) is
ready to attach to the next notebook. Each notebook's output contains every earlier stage, so you only ever attach
the previous notebook.

In [1]:
# ── Setup: make the factindic code importable ─────────────────────────────────────────
# Kaggle: upload factindic-code.zip as a private dataset and attach it
#         (right-hand panel -> Add Input -> Your Work / Datasets -> factindic-code -> "+").
# Colab:  put factindic-code.zip in My Drive (and run the Google Drive cell first), or upload it to /content.
# Or:     push the repository to GitHub and paste its URL here (needs Internet).
REPO_URL = ""   # e.g. "https://github.com/<your-username>/factindic"

import glob, os, shutil, subprocess, sys, zipfile

ON_COLAB = "google.colab" in sys.modules
SRC = "/content/factindic_src" if ON_COLAB else "/kaggle/working/factindic_src"

def _is_pkg(d):
    return all(os.path.exists(os.path.join(d, f)) for f in ("__init__.py", "pipeline.py", "config.py"))

def _candidates():
    inits = glob.glob("/kaggle/input/**/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if ON_COLAB:  # shallow search only: Google Drive can be huge
        for base in ("/content", "/content/drive/MyDrive"):
            for depth in ("", "*/", "*/*/"):
                inits += glob.glob(f"{base}/{depth}factindic/__init__.py")
            zips += [z for z in glob.glob(f"{base}/*.zip") + glob.glob(f"{base}/*/*.zip")
                     if "factindic" in os.path.basename(z).lower()]
    return sorted(set(inits)), sorted(set(zips))

def _unzip(z):
    try:
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith("factindic/__init__.py")]
            if hits:
                shutil.rmtree(SRC, ignore_errors=True)
                zf.extractall(SRC)
                return os.path.dirname(os.path.dirname(os.path.join(SRC, sorted(hits, key=len)[-1])))
    except zipfile.BadZipFile:
        pass
    return None

def _find_code():
    inits, zips = _candidates()
    # 0) Colab: always unpack the newest zip from Drive, so replacing the zip there updates the code
    if ON_COLAB:
        for z in sorted(zips, key=os.path.getmtime, reverse=True):
            code = _unzip(z)
            if code:
                print("using", z)
                return code
    # 1) the repository folder (…/factindic/factindic/__init__.py)
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d) and os.path.basename(d) == "factindic":
            return os.path.dirname(d)
    # 2) only the inner package was uploaded (files directly in the dataset): copy it under the right name
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d):
            shutil.copytree(d, os.path.join(SRC, "factindic"), dirs_exist_ok=True)
            for req in glob.glob(os.path.join(os.path.dirname(d), "**", "requirements.txt"), recursive=True)[:1]:
                shutil.copy(req, SRC)
            return SRC
    # 3) a zip that nobody unpacked
    for z in zips:
        code = _unzip(z)
        if code:
            return code
    # 4) running locally from the repository
    for p in ("factindic", "../factindic"):
        if _is_pkg(p):
            return os.path.abspath(os.path.dirname(p) or ".")
    return None

CODE = _find_code()
if CODE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, SRC], check=True)
    CODE = SRC
if CODE is None:
    where = (sorted(glob.glob("/content/*")) + sorted(glob.glob("/content/drive/MyDrive/*"))[:30]) if ON_COLAB else \
            (sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30])
    how = ("Put factindic-code.zip in My Drive and run the Google Drive cell first (or upload the zip to /content)."
           if ON_COLAB else
           "Attach the dataset: right-hand panel -> Add Input -> Your Work (or Datasets) -> factindic-code -> '+'. "
           "If you uploaded it a minute ago, wait until Kaggle finishes processing it.")
    raise FileNotFoundError("factindic code not found.\n" + how + "\nWhat this notebook can see:\n  "
                            + ("\n  ".join(where) if where else "(nothing)"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
req = os.path.join(CODE, "requirements.txt")
pkgs = ["-r", req] if os.path.exists(req) else ["indic-transliteration", "wordfreq", "datasketch", "bert-score",
                                                "python-docx", "feedparser", "trafilatura", "peft", "openpyxl",
                                                "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import factindic
from factindic import Config
from factindic import pipeline as P
print("factindic", factindic.__version__, "loaded from", CODE)

factindic 1.0.0 loaded from /kaggle/input/datasets/lkr1196/factindic-code/factindic


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Use the SAME profile in every notebook of the chain.
#   "full"  – the real experiment
#   "quick" – ~25 articles per language, to check the whole chain on Kaggle first
#   "mock"  – CPU dry run with stand-in models (no GPU, no downloads); never report its numbers
PROFILE = "full"
cfg = Config.make(PROFILE)
# cfg.codemix_model = "Qwen/Qwen3-8B"   # a different, larger rewriter: about 4x slower
print(cfg.to_json())

{
  "profile": "full",
  "seed": 13,
  "langs": [
    "hi",
    "bn"
  ],
  "variants": [
    "mono",
    "cm",
    "roman"
  ],
  "data_source": "xlsum",
  "n_test": 200,
  "n_dev": 30,
  "n_train_cm": 60,
  "min_words": 120,
  "max_words_raw": 1200,
  "lead_words": 300,
  "max_sents": 14,
  "min_summary_words": 8,
  "max_summary_words": 60,
  "near_dup_threshold": 0.8,
  "natural_split": [
    0.7,
    0.1,
    0.2
  ],
  "scrape": true,
  "n_natural_per_lang": 100,
  "backend": "auto",
  "summarizer_model": "Qwen/Qwen3-4B-Instruct-2507",
  "codemix_model": "Qwen/Qwen3-4B-Instruct-2507",
  "openai_base_url": "",
  "openai_model": "",
  "openai_concurrency": 4,
  "max_model_len": 6144,
  "max_batch_tokens": 24000,
  "max_batch_size": 24,
  "repetition_penalty": 1.0,
  "workers": -1,
  "quantize": "auto",
  "summary_max_words": 40,
  "tracks": [
    "A",
    "B",
    "BV"
  ],
  "max_new_tokens_A": 160,
  "max_new_tokens_B": 512,
  "seq2seq_models": [
    "mt5",
    "indicbart"
  ],
  

In [3]:
var = P.run_stage02(cfg)

Stages available: 01_data
Rewriting 400 test and 120 training articles
cm: 1394 prompts for 520 documents


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Starting 2 worker(s) for 1394 prompts with Qwen/Qwen3-4B-Instruct-2507 (logs in /kaggle/working/factindic_out/_jobs/rewrite_cm_639a6b63)
  [worker 0] Loading Qwen/Qwen3-4B-Instruct-2507 for 697 prompts
  [worker 1] Loading Qwen/Qwen3-4B-Instruct-2507 for 697 prompts
  [worker 1] Loading weights: 100%|██████████| 398/398 [00:07<00:00, 50.85it/s]
  [worker 0] worker 0: 24/697 generated (3.7 min, ~105 min left; avg batch 4.9, 35 tokens/s)
  [worker 1] worker 1: 24/697 generated (4.6 min, ~129 min left; avg batch 5.1, 27 tokens/s)
  [worker 0] worker 0: 24/697 generated (3.7 min, ~105 min left; avg batch 4.9, 35 tokens/s)
  [worker 1] worker 1: 24/697 generated (4.6 min, ~129 min left; avg batch 5.1, 27 tokens/s)
  [worker 0] worker 0: 24/697 generated (3.7 min, ~105 min left; avg batch 4.9, 35 tokens/s)
  [worker 1] worker 1: 24/697 generated (4.6 min, ~129 min left; avg batch 5.1, 27 tokens/s)
  [worker 0] worker 0: 24/697 generated (3.7 min, ~105 min left; avg batch 4.9, 35 tokens/s)
  

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Starting 2 worker(s) for 93 prompts with Qwen/Qwen3-4B-Instruct-2507 (logs in /kaggle/working/factindic_out/_jobs/rewrite_roman_retry_68933fc3)
  [worker 0] Loading Qwen/Qwen3-4B-Instruct-2507 for 47 prompts
  [worker 1] Loading Qwen/Qwen3-4B-Instruct-2507 for 46 prompts
  [worker 0] worker 0: 24/47 generated (0.3 min, ~0 min left; avg batch 14.6, 58 tokens/s)
  [worker 1] worker 1: 46/46 generated (0.6 min, ~0 min left; avg batch 12.9, 58 tokens/s)


,lang,role,variant,docs,valid,fallback_sents,cmi,en_share,switch_rate,latin_chars,gen_min
0,bn,natural,natural,61,1.000000,0,0.980859,0.015647,0.005061,0.015918,0.000000
1,bn,test,cm,200,0.863125,369,18.371254,0.183713,0.234347,0.214650,138.140698
2,bn,test,mono,200,1.000000,0,0.031114,0.000311,0.000131,0.000238,0.000000
3,bn,test,roman,200,0.998516,4,28.435467,0.303155,0.319524,0.999680,36.684195
4,bn,train,cm,60,0.861255,113,17.926073,0.179261,0.229799,0.210282,43.674925
5,bn,train,mono,60,1.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000
6,bn,train,roman,60,0.995238,4,30.250574,0.319672,0.314241,0.999503,11.460350
7,hi,natural,natural,72,1.000000,0,1.999742,0.019997,0.014901,0.022517,0.000000
8,hi,test,cm,200,0.641933,851,9.562849,0.095628,0.130775,0.146019,112.090312
9,hi,test,mono,200,1.000000,0,0.016855,0.000169,0.000339,0.000254,0.000000


In [4]:
from IPython.display import Markdown, display
display(Markdown(open(f"{cfg.out_root}/02_codemix/codemix_report.md", encoding="utf-8").read()))

# Code-mixed variants

| lang | role | variant | docs | valid | fallback_sents | cmi | en_share | switch_rate | latin_chars | gen_min |
|---|---|---|---|---|---|---|---|---|---|---|
| bn | natural | natural | 61 | 1.00 | 0 | 0.98 | 0.02 | 0.01 | 0.02 | 0.00 |
| bn | test | cm | 200 | 0.86 | 369 | 18.37 | 0.18 | 0.23 | 0.21 | 138.14 |
| bn | test | mono | 200 | 1.00 | 0 | 0.03 | 0.00 | 0.00 | 0.00 | 0.00 |
| bn | test | roman | 200 | 1.00 | 4 | 28.44 | 0.30 | 0.32 | 1.00 | 36.68 |
| bn | train | cm | 60 | 0.86 | 113 | 17.93 | 0.18 | 0.23 | 0.21 | 43.67 |
| bn | train | mono | 60 | 1.00 | 0 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 |
| bn | train | roman | 60 | 1.00 | 4 | 30.25 | 0.32 | 0.31 | 1.00 | 11.46 |
| hi | natural | natural | 72 | 1.00 | 0 | 2.00 | 0.02 | 0.01 | 0.02 | 0.00 |
| hi | test | cm | 200 | 0.64 | 851 | 9.56 | 0.10 | 0.13 | 0.15 | 112.09 |
| hi | test | mono | 200 | 1.00 | 0 | 0.02 | 0.00 | 0.00 | 0.00 | 0.00 |
| hi | test | roman | 200 | 0.99 | 14 | 19.63 | 0.20 | 0.27 | 1.00 | 31.53 |
| hi | train | cm | 60 | 0.67 | 242 | 10.09 | 0.10 | 0.14 | 0.15 | 36.41 |
| hi | train | mono | 60 | 1.00 | 0 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 |
| hi | train | roman | 60 | 0.99 | 4 | 19.38 | 0.19 | 0.28 | 1.00 | 9.39 |

## Share of test articles usable for paired comparison

| lang | paired_ok_share |
|---|---|
| bn | 0.71 |
| hi | 0.30 |

Total time: 215.7 min


Three versions of the same sentence:

In [5]:
t = var[var.role == "test"]
for doc_id in t.doc_id.unique()[:3]:
    g = t[t.doc_id == doc_id].set_index("variant")
    for v in ("mono", "cm", "roman"):
        if v in g.index:
            print(f"{v:6s}", g.at[v, "sents"][0])
    print()

mono   दिवंगत प्रधानमंत्री राजीव गाँधी 1987 में बर्मा गए थे.
cm     दिवंगत प्रधानमंत्री राजीव गाँधी 1987 में Myanmar गए थे।
roman  Divangat pradhanmantri Rajiv Gandhi 1987 mein Myanmar gae the.

mono   'कितने पाकिस्तान' उपन्यास एक बड़ा और नए ढंग का उपन्यास है.
cm     'कितने पाकिस्तान' उपन्यास एक बड़ा और नए ढंग का उपन्यास है.
roman  Kitanे Pakistan upanyas ek bda aur nayi dhang ka upanyas hai.

mono   उप प्रधानमंत्री आडवाणी ने ऑटो एक्सपो 2004 का उदघाटन करते हुए ये बात कही.
cm     उप प्रधानमंत्री आडवाणी ने ऑटो एक्सपो 2004 का उदघाटन करते हुए ये बात कही।
roman  Up pradhanmantri Aadwani ne auto expo 2004 ka udghatan karte hue ye baat kahi.



In [6]:
import os
for root, dirs, files in os.walk(cfg.out_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    depth = root[len(cfg.out_root):].count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/", ", ".join(sorted(files))[:300])

factindic_out/ 
  02_codemix/ codemix_quality_check.xlsx, codemix_report.md, config.json, env.json, variants.jsonl
  01_data/ config.json, data_report.md, data_stats.csv, docs.jsonl
